# Bonus Challenge 5: Deploying Agents to Agent Engine
Deploy an ADK agent to Google Agent Engine and test it remotely.

In [ ]:
!pip install google-adk google-cloud-aiplatform[agent_engines,adk]

## Setup imports and environment

In [ ]:
import os
import vertexai
from google.adk import Agent
from google.adk.tools import google_search
from vertexai.preview.reasoning_engines import AdkApp
from vertexai import agent_engines

PROJECT_ID = "qwiklabs-gcp-01-fab96dd167ae"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://YOUR_STAGING_BUCKET"  # Create a GCS bucket and put its name here

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

## Define the agent
Using a simple search agent for deployment.

In [ ]:
search_agent = Agent(
    name="search_agent",
    model="gemini-2.0-flash-lite",
    description="A search agent that answers questions using Google Search.",
    instruction="""You are a helpful assistant. Use Google Search to find accurate,
up-to-date information and answer the user's question clearly and concisely.""",
    tools=[google_search],
)

## Step 1: Test the agent locally
Wrap in AdkApp and test with stream_query before deploying.

In [ ]:
app = AdkApp(agent=search_agent)

for event in app.stream_query(
    user_id="test_user",
    message="What are the latest headlines in technology?",
):
    print(event)

## Step 2: Deploy to Agent Engine
This takes approximately 10 minutes.

In [ ]:
remote_agent = agent_engines.create(
    app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]"],
)

print(f"Deployed! Resource name: {remote_agent.resource_name}")

## Step 3: Test the deployed agent
Query the remote agent to prove it works.

In [ ]:
for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="What is the current weather news in the United States?",
):
    print(event)

## Step 4: Test with another query

In [ ]:
for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="What are the top 3 AI developments this week?",
):
    print(event)

## Cleanup (optional)
Delete the deployed agent when done to avoid charges.

In [ ]:
# Uncomment to delete:
# remote_agent.delete()